# GenCultura — Model Server on Colab

Runs two OpenAI-compatible servers:
- **Port 8000** — vLLM serving the LLM (chat completions)
- **Port 8001** — infinity-emb serving the embedding model

Both are tunnelled to public URLs via ngrok.  
At the end, copy the printed `.env` block into your `genCultura/.env`.

**Runtime:** Runtime → Change runtime type → T4 GPU (free tier is fine for 3–7B models)

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout or 'No GPU detected — switch runtime to GPU')

In [ ]:
# ── Cell 2: Install packages ─────────────────────────────────────────────────
# vLLM: LLM inference server
# infinity-emb: fast OpenAI-compatible embedding server
# pyngrok: tunnel local ports to public HTTPS URLs
%pip install -q vllm
%pip install -q "infinity-emb[all]"
%pip install -q pyngrok
print('Done.')

In [ ]:
# ── Cell 3: ngrok auth ───────────────────────────────────────────────────────
# Free account at https://ngrok.com — your token is at https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok

NGROK_TOKEN = ""  # ← paste your ngrok auth token here

assert NGROK_TOKEN, "Paste your ngrok auth token above."
ngrok.set_auth_token(NGROK_TOKEN)
print('ngrok token set.')

In [ ]:
# ── Cell 4: Model selection ──────────────────────────────────────────────────
# LLM — choose based on available VRAM:
#   T4 (15 GB):  Qwen/Qwen2.5-7B-Instruct  (needs --dtype half, ~8 GB)
#                Qwen/Qwen2.5-3B-Instruct  (safe choice, ~3.5 GB)
#   A100 (40 GB): mistralai/Mistral-7B-Instruct-v0.3, meta-llama/Llama-3.1-8B-Instruct
#
# Embedding — nomic-embed-text (768-dim) is a strong open-source choice.
# NOTE: if you change EMBEDDING_MODEL away from OpenAI's text-embedding-3-small (1536-dim)
#       you must also update EMBEDDING_DIMENSIONS in your .env and re-run Alembic migrations.

LLM_MODEL       = "Qwen/Qwen2.5-3B-Instruct"          # change to 7B if you have Colab Pro
EMBEDDING_MODEL = "nomic-ai/nomic-embed-text-v1.5"     # 768-dim
EMBEDDING_DIMS  = 768

LLM_PORT   = 8000
EMBED_PORT = 8001

print(f'LLM:       {LLM_MODEL}')
print(f'Embedding: {EMBEDDING_MODEL}  ({EMBEDDING_DIMS} dims)')

In [ ]:
# ── Cell 5: Start embedding server (infinity-emb) ────────────────────────────
import subprocess, time, requests

embed_proc = subprocess.Popen(
    [
        'infinity_emb', 'v2',
        '--model-name-or-path', EMBEDDING_MODEL,
        '--port', str(EMBED_PORT),
        '--served-model-name', EMBEDDING_MODEL,
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print(f'Waiting for embedding server on port {EMBED_PORT}...')
for _ in range(60):
    try:
        r = requests.get(f'http://localhost:{EMBED_PORT}/health', timeout=2)
        if r.status_code == 200:
            print('Embedding server ready.')
            break
    except Exception:
        pass
    time.sleep(3)
else:
    # Print last output to debug
    out, _ = embed_proc.communicate(timeout=1)
    print('Timed out. Server output:')
    print(out.decode() if out else '(none)')

In [ ]:
# ── Cell 6: Start LLM server (vLLM) ─────────────────────────────────────────
import subprocess, time, requests

llm_proc = subprocess.Popen(
    [
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', LLM_MODEL,
        '--port', str(LLM_PORT),
        '--dtype', 'half',                  # fp16 — required on T4
        '--max-model-len', '4096',          # limit context to save VRAM
        '--gpu-memory-utilization', '0.85', # leave headroom for embedding server
        '--served-model-name', LLM_MODEL,
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print(f'Waiting for vLLM on port {LLM_PORT} (model download may take a few minutes)...')
for _ in range(120):
    try:
        r = requests.get(f'http://localhost:{LLM_PORT}/health', timeout=2)
        if r.status_code == 200:
            print('vLLM ready.')
            break
    except Exception:
        pass
    time.sleep(5)
else:
    out, _ = llm_proc.communicate(timeout=1)
    print('Timed out. Server output:')
    print(out.decode()[-3000:] if out else '(none)')

In [ ]:
# ── Cell 7: Open ngrok tunnels and print .env block ──────────────────────────
from pyngrok import ngrok

llm_tunnel   = ngrok.connect(LLM_PORT,   bind_tls=True)
embed_tunnel = ngrok.connect(EMBED_PORT, bind_tls=True)

llm_url   = llm_tunnel.public_url
embed_url = embed_tunnel.public_url

print('\n' + '='*60)
print('PASTE THIS INTO YOUR genCultura/.env')
print('='*60)
print(f"""
# ── Colab model servers ──────────────────────────────────────
# LLM
OPENAI_BASE_URL={llm_url}/v1
OPENAI_API_KEY=no-key
LLM_MODEL={LLM_MODEL}

# Embedding  ← also update backend/alembic/versions/0001_initial_schema.py
#               EMBEDDING_DIMS = {EMBEDDING_DIMS}  and re-run: alembic downgrade base && alembic upgrade head
EMBEDDING_BASE_URL={embed_url}/v1
EMBEDDING_MODEL={EMBEDDING_MODEL}
EMBEDDING_DIMENSIONS={EMBEDDING_DIMS}
""")
print('='*60)
print(f'\nTunnels active:')
print(f'  LLM:       {llm_url}')
print(f'  Embedding: {embed_url}')
print('\nKeep this Colab tab open. Tunnels close when the session ends.')

In [ ]:
# ── Cell 8: Smoke test both endpoints ───────────────────────────────────────
import requests, json

# Test LLM
print('--- LLM test ---')
r = requests.post(
    f'{llm_url}/v1/chat/completions',
    json={
        'model': LLM_MODEL,
        'messages': [{'role': 'user', 'content': 'Say "LLM OK" and nothing else.'}],
        'max_tokens': 10,
    }
)
print(r.json()['choices'][0]['message']['content'])

# Test embedding
print('\n--- Embedding test ---')
r = requests.post(
    f'{embed_url}/v1/embeddings',
    json={'model': EMBEDDING_MODEL, 'input': 'test observation'}
)
vec = r.json()['data'][0]['embedding']
print(f'Vector dims: {len(vec)}  (expected {EMBEDDING_DIMS})')
print('Both endpoints working.')

In [ ]:
# ── Cell 9: Tear down (run when done) ────────────────────────────────────────
# ngrok.disconnect(llm_url)
# ngrok.disconnect(embed_url)
# llm_proc.terminate()
# embed_proc.terminate()
# print('Servers stopped.')